# Example filtering (Screws and wall plugs)

In [ ]:
# Parameters
RUN_ID = 19
SEED = 42


In [ ]:
import torch
import torch.nn as nn
import torch.nn.init as init
import numpy as np
from mtlearn import morphology
import cv2 as cv
import mtlearn
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
from scipy.special import expit as sigmoid
import time
import os, random
import matplotlib.pyplot as plt

def showImages(ds, n=1):
    # Exemplo de uso: showImages(trainset.dataset, 5)
    plt.figure(figsize=(6, 3 * n))
    for i in range(n):
        row = ds.__getitem__(i)
        x, y = row[0], row[1]
        imgInput = x.squeeze().detach().numpy()
        imgOutput = y.squeeze().detach().numpy()

        #imgInput = np.moveaxis(imgInput, 0, -1)
        # Subplot da entrada (coluna 1)
        plt.subplot(n, 2, 2*i + 1)
        plt.imshow(imgInput, cmap='gray')
        plt.title(f"Input {i}")
        plt.axis('off')

        # Subplot da saída (coluna 2)
        plt.subplot(n, 2, 2*i + 2)
        plt.imshow(imgOutput, cmap='gray')
        plt.title(f"Output {i}")
        plt.axis('off')



    plt.tight_layout()
    plt.show()

def showPredict(data, model, threshold, model_base, threshold_base):
    model.eval()
    with torch.no_grad():
        for i in range(len(data)):
            x, y, name = data[i]
            y = y.cpu().numpy().flatten()
            x = x.unsqueeze(1)
            logits = model(x)[0]
            logits_base = model_base(x)[0]

            filterCC = model.morphological_layer(x).cpu()
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            probs_base = torch.sigmoid(logits_base).cpu().numpy().flatten()
            y_pred = (probs > threshold).astype(int)
            y_pred_base = (probs_base > threshold_base).astype(int)
            print(f"{name}")
            print(f"MAE (without CFP): {np.mean( y != y_pred_base ):.3f},\t Recall (without CFP): \t{recall_score(y, y_pred_base):.3f},\t Precision (without CFP): {precision_score(y,y_pred_base):.3f},\t Jaccard (without CFP): {jaccard_score(y, y_pred_base):.3f}")
            print(f"MAE (with CFP): {np.mean( y != y_pred ):.3f},\t\t Recall (with CFP): \t{recall_score(y, y_pred):.3f},\t Precision (with CFP): \t{precision_score(y,y_pred):.3f},\t\t Jaccard (with CFP): \t{jaccard_score(y, y_pred):.3f}")

            input_img = x.cpu().reshape(num_rows, num_cols).numpy()
            target = y.reshape(num_rows, num_cols)*255
            filter_output = filterCC.reshape(num_rows, num_cols).numpy()
            filter_output = (filter_output - filter_output.min()) / (filter_output.max() - filter_output.min()) * 255
            output = y_pred.reshape(num_rows, num_cols)*255
            output_base = y_pred_base.reshape(num_rows, num_cols)*255

            plt.figure(figsize=(25, 9))
            plt.subplot(1,5,1)
            plt.imshow(input_img, cmap='gray')
            plt.title('Input')

            plt.subplot(1,5,2)
            plt.imshow(target, cmap='gray', vmax=255, vmin=0)
            plt.title('Target')

            plt.subplot(1,5,3)
            plt.imshow(filter_output, cmap='gray')
            plt.title('Output CFP layer')

            plt.subplot(1,5,4)
            plt.imshow(output, cmap='gray', vmax=255, vmin=0)
            plt.title('Predict (with CFP)')

            plt.subplot(1,5,5)
            plt.imshow(output_base, cmap='gray', vmax=255, vmin=0)
            plt.title('Predict (without CFP)')
            plt.show()



def fix_randomness(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        # cuDNN e escolha de algoritmos determinísticos
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)

        # Algumas operações GEMM em GPU precisam disso para *bitwise* determinism
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # ou ":16:8"

fix_randomness(SEED)
generator = torch.Generator(device="cpu")
generator.manual_seed(SEED)

In [ ]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

# Dataset (trainset and testset)

In [ ]:
num_cols, num_rows =(int(1177 * 0.5), int(1324 * 0.499))
print("Resolution: ", num_cols, "x", num_rows)
dir_name = str(mtlearn.data.ensure_dataset("screws_segmentation"))
dataset = mtlearn.datasets.PairedImageDataset(root_dir=dir_name, grayscale_in=True, invert_in=True, invert_target=True,  numRows=num_rows, numCols=num_cols, scale_in=False, suffix_in="_in", suffix_target="_target")

trainset, testset = dataset.train_test_split(test_size=0.3, random_state=SEED)
print("trainset:", len(trainset))
print("testset:", len(testset))
showImages(trainset, 2)

#transferindo o conjunto de treinamento/teste para o device
trainset = [(entradas.to(device), saidas.to(device), nameFiles) for entradas, saidas, nameFiles in trainset]
testset= [(entradas.to(device), saidas.to(device), nameFiles) for entradas, saidas, nameFiles in testset]
trainloader = torch.utils.data.DataLoader(trainset, batch_size=8, shuffle=True, pin_memory=False, generator=generator)


In [ ]:
# Calcula a razão de desbalanceamento
num_neg = []
num_pos = []
for i in range(len(trainset)):
    x, y, name = trainset[i]
    num_neg.append((y == 0).sum().item())
    num_pos.append((y == 1).sum().item())

# Calcula as proporções
total_neg = sum(num_neg)
total_pos = sum(num_pos)
total_samples = total_neg + total_pos

percent_neg = (total_neg / total_samples) * 100
percent_pos = (total_pos / total_samples) * 100

print(f"Total de amostras classe 0: {total_neg} ({percent_neg:.2f}%)")
print(f"Total de amostras classe 1: {total_pos} ({percent_pos:.2f}%)")

pos_weight = torch.tensor([total_neg / total_pos])  # 8498057 / 530683 ≈ 16
print(f"pos_weight: {pos_weight}")

# Connected filter preprocessing layer

In [ ]:
Type = morphology.AttributeType
tree_type = "max-tree"
cfp_layer = mtlearn.layers.ConnectedFilterPreprocessingLayer(
    in_channels=1,
    attributes_spec=[
                      ( #output channel 1
                        Type.AREA,
                        Type.LENGTH_MAJOR_AXIS,
                        Type.LENGTH_MINOR_AXIS,
                        Type.INERTIA,
                        Type.CIRCULARITY,
                        Type.GRAY_HEIGHT,
                        Type.RECTANGULARITY,
                        Type.MAX_DIST,
                      )
                      #,
                      # ( #output channel 2
                      #    Type.AREA,
                      #    Type...
                      # )
    ],
    tree_type=tree_type,
    device=device
)

#Cache trainset and initialize filter as identity
trainloader_cached = cfp_layer.build_dataloader_cached(trainloader)
cfp_layer.init_identity_with_bias()


"""
If you wish, attach the CFP to a backbone. For example:

class BaseNet(nn.Module):
    def __init__(self):
        super(BaseNet, self).__init__()

        # === A simple ConvNet ===
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(8, 1, kernel_size=3, padding=1)
        ).to(device)
        #initialize parameters...

    def forward(self, inputs):
        return self.net(inputs)


class UnetWithConnectedFilterPreprocessingLayer(nn.Module):
    def __init__(self, cfp_layer):
        super(UnetWithConnectedFilterPreprocessingLayer, self).__init__()
        self.cfp_layer = cfp_layer
        self.net = BaseNet()

    def forward(self, inputs):
        return self.net( self.cfp_layer(inputs) )


"""

print("The CFP layer has been initialized.")

# Training

In [ ]:
num_epochs = 100
learning_rate = 0.05

loss_function = nn.MSELoss(reduction="sum" ) #reduction="sum" <==> loss = (num_rows*num_cols) * loss_function(predicts, targets) #with reduction="mean"
optimizer = torch.optim.Adam(cfp_layer.parameters(), lr=learning_rate, weight_decay=1e-7)
errors = []
cfp_layer.train()
time_begin = time.time()
for epoch in range(num_epochs):
    epoch_loss = 0
    for i, (inputs, targets) in enumerate(trainloader_cached):


        # Forward
        predicts = cfp_layer(inputs)

        # Loss
        loss =  loss_function(predicts, targets)
        
        epoch_loss += loss.item()

        # Backward + Optimize
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cfp_layer.parameters(), max_norm=5.0)
        optimizer.step()

    errors.append(epoch_loss / len(trainloader))
    if(epoch % 20 == 0 or epoch == num_epochs-1):
        print(f"Epoch {epoch}, Loss: {errors[-1]:.7f}")

        (imgs_in, keys_in), imgs_out = inputs, targets.cpu()
        
        i = np.random.randint(0, len(imgs_out))
        fig, ax = plt.subplots(1, 3, figsize=(15, 7))
        ax[0].imshow(imgs_in[i,0].cpu().numpy(), cmap="gray");  ax[0].set_title("Input")
        ax[1].imshow(imgs_out[i,0].numpy(), cmap="gray"); ax[1].set_title("Target")
        ax[2].imshow(predicts[i,0].cpu().detach().numpy(), cmap="gray");    ax[2].set_title("Connected filter")
        #ax[3].imshow(imgs_pred[i,0], cmap="gray");    ax[3].set_title("Pred")
        for a in ax: a.axis("off")
        plt.tight_layout(); plt.show()



time_end = time.time()
print(f"Execution time: {(time_end - time_begin)/60:.3f} minutes")


plt.figure(figsize=(12, 5))
plt.plot(errors, '-')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Evolution of loss')
plt.show()

# Predict

In [ ]:
def showPredict(data, model):
    model.eval()
    with torch.no_grad():
      for i in range(len(data)):
          inputs, targets, name = data[i]
          inputs = inputs.unsqueeze(0)
          targets = targets.unsqueeze(0)
          predicts = model(inputs)

          fig, ax = plt.subplots(1, 3, figsize=(15, 7))
          ax[0].imshow(inputs[0,0].cpu().numpy(), cmap="gray");  ax[0].set_title(f"Input {name}")
          ax[1].imshow(targets[0,0].cpu().numpy(), cmap="gray"); ax[1].set_title("Target")
          ax[2].imshow(predicts[0,0].cpu().detach().numpy(), cmap="gray");    ax[2].set_title("Connected filter")
          for a in ax: a.axis("off")
          plt.tight_layout(); plt.show()


showPredict(testset, cfp_layer)